This script is an ad hoc request to create a census file with msa names for 2024 before the census 2024 flat file is ready.

The file is needed earlier than the normal ~September timeframe so GEOIDs register correctly on the platform.  

The census 2024 should include the new Connecticut county codes with their respective MSA Name and ID.  

*Caveats
Anything other than GEO ID information is inaccurate

# Setup

In [24]:
import pandas as pd
import os
import datacompy
import numpy as np

# Import Data

In [25]:
df_delin_2022 = pd.read_excel("../data/census_data/excel_delineation_2022.xls",skiprows=2,index_col=None,dtype=object, engine="xlrd").dropna(subset=['CBSA Code','FIPS State Code'])
df_delin_2023_raw = pd.read_excel("../data/census_data/excel_delineation_2023.xlsx",skiprows=2,index_col=None,dtype=object).dropna(subset=['CBSA Code','FIPS State Code'])

In [26]:
df_census_2023 = pd.read_csv("../output/ffiec_census_msamd_names_2023.txt",delimiter="|", dtype=object )
df_census_2023["KEY"] = df_census_2023["State"] + df_census_2023["County"] + df_census_2023["Census Tract"] 
df_census_2024_raw = df_census_2023.copy()
df_census_2024_raw["Collection Year"] = "2024"

df_CT_map = pd.read_csv("../output_FHFA/connecticut_county_code_mapping.csv", dtype=object)


In [27]:
delin_keep_cols = ['CBSA Code', 'CBSA Title','FIPS State Code', 'FIPS County Code' ]

df_delin_2023 = df_delin_2023_raw[delin_keep_cols]

In [28]:
df_delin_2023 = df_delin_2023.merge(df_CT_map,  how = "inner", left_on = ['FIPS State Code', 'FIPS County Code'],right_on = ['STATEFP', 'old_county_code'],suffixes=('_2023',""))


df_delin_2023["COUNTY_CODE"] = df_delin_2023["new_county_code"].fillna(df_delin_2023['FIPS County Code'] )
df_delin_2023["STATE"] = df_delin_2023['FIPS State Code']
df_delin_2023["MSA_NAME"] = df_delin_2023['CBSA Title']
df_delin_2023["MSA_ID"] = df_delin_2023['CBSA Code']
df_delin_2023["TRACT"] = df_delin_2023["NAMELSAD"]


In [29]:
df_delin_2023 = df_delin_2023[['old_county_code', 'new_county_code',
       'COUNTY_CODE', 'STATE', 'MSA_NAME', 'MSA_ID', 'TRACT']]
df_delin_CT_2023 = df_delin_2023[(df_delin_2023['STATE' ] == '09')].copy()
df_delin_CT_2023 = df_delin_2023

In [30]:
df_census_2024_temp = df_census_2024_raw.copy()

# set up KEY

In [31]:

df_census_2024_temp["KEY"] = df_census_2024_temp["State"] + df_census_2024_temp["County"] + df_census_2024_temp["Census Tract"] 


df_delin_CT_2023["Census Tract"] = df_delin_CT_2023["TRACT"].replace("Census Tract ","",regex= True)
df_delin_CT_2023[["Census Tract", "Census Tract2"]] = df_delin_CT_2023["Census Tract"].str.split(".",expand=True).fillna("00")  # split on "." and fill with "00" ,   # This "right fills" 
df_delin_CT_2023["Census Tract"] = df_delin_CT_2023["Census Tract"] + df_delin_CT_2023["Census Tract2"] # rejoin 
df_delin_CT_2023["Census Tract"] = df_delin_CT_2023["Census Tract"].str.zfill(6) #left fill with 0


df_delin_CT_2023["KEY"] = df_delin_CT_2023["STATE"] + df_delin_CT_2023["old_county_code"] + df_delin_CT_2023["Census Tract"]

In [32]:
df_delin_CT_2023_map = df_delin_CT_2023[['old_county_code', 'new_county_code', 'STATE',
       'MSA_NAME','TRACT', 'Census Tract', 'KEY']]

## create maps

In [33]:
# Conneticut old to New County Codes
df_map = df_delin_CT_2023[["KEY", "old_county_code","new_county_code","TRACT",'MSA_NAME', 'MSA_ID',"Census Tract"]].drop_duplicates()
map_county_code = dict(zip(df_map["KEY"], df_map["new_county_code"]))
map_mds_name_code = dict(zip(df_map["KEY"], df_map["MSA_NAME"]))
map_mds_id_code = dict(zip(df_map["KEY"], df_map["MSA_ID"]))
map_tract_code = dict(zip(df_map["KEY"], df_map["Census Tract"]))



In [34]:

df_census_2024_temp["COUNTY_2024"] = df_census_2024_temp["KEY"].map(map_county_code)
df_census_2024_temp["COUNTY_2024"] = df_census_2024_temp["COUNTY_2024"].fillna(df_census_2024_temp["County"])

df_census_2024_temp["MSA_NAME_2024"] = df_census_2024_temp["KEY"].map(map_mds_name_code)
df_census_2024_temp["MSA_NAME_2024"] = df_census_2024_temp["MSA_NAME_2024"].fillna(df_census_2024_temp['MSA/MD Name'])

df_census_2024_temp["MSA_ID_2024"] = df_census_2024_temp["KEY"].map(map_mds_id_code)
df_census_2024_temp["MSA_ID_2024"] = df_census_2024_temp["MSA_ID_2024"].fillna(df_census_2024_temp['MSA/MD'])

df_census_2024_temp["TRACT_2024"] = df_census_2024_temp["KEY"].map(map_tract_code)
df_census_2024_temp["TRACT_2024"] = df_census_2024_temp["TRACT_2024"].fillna(df_census_2024_temp["Census Tract"])



In [35]:
df_census_2024_temp.isna().sum()


Collection Year                       0
MSA/MD                                0
State                                 0
County                                0
Census Tract                          0
FFIEC Median Family Income            0
Population                            2
Minority Population %               133
Number of Owner Occupied Units        2
Number of 1 to 4 Family Units       133
Tract MFI                            23
Tract to MSA Income %                 0
Median Age                          133
Small County                          0
MSA/MD Name                       15403
KEY                                   0
COUNTY_2024                           0
MSA_NAME_2024                     15351
MSA_ID_2024                           0
TRACT_2024                            0
dtype: int64

In [36]:
df_census_2024_final = df_census_2024_temp.copy()
df_census_2024_final["County"] = df_census_2024_final["COUNTY_2024"]
df_census_2024_final['MSA/MD Name'] = df_census_2024_final["MSA_NAME_2024"] 
df_census_2024_final['MSA/MD'] = df_census_2024_final["MSA_ID_2024"] 
df_census_2024_final["Census Tract"] = df_census_2024_final["TRACT_2024"]


df_census_2024_final = df_census_2024_final[["KEY",'Collection Year', 'MSA/MD', 'State', 'County', 'Census Tract',
       'FFIEC Median Family Income', 'Population', 'Minority Population %',
       'Number of Owner Occupied Units', 'Number of 1 to 4 Family Units',
       'Tract MFI', 'Tract to MSA Income %', 'Median Age', 'Small County',
       'MSA/MD Name']]

df_census_2024_final = df_census_2024_final.drop_duplicates()

In [37]:
df_ct_2024_all= df_census_2024_final[(df_census_2024_final['State' ] == '09')]


df_ct_2024_all_old = df_census_2024_final[(df_census_2024_final['State' ] == '09') & (df_census_2024_final['County'].astype(int) <= 15)]
df_ct_2024_all_new = df_census_2024_final[(df_census_2024_final['State' ] == '09') & (df_census_2024_final['County'].astype(int) > 15)]

In [38]:
# df_compare = datacompy.Compare(df_census_2024_final, df_census_2023,join_columns=["Census Tract", "State", "County"])


df_compare = datacompy.Compare(df_census_2024_final[["KEY",'Collection Year', 'MSA/MD', 'State', 'County', 'Census Tract','MSA/MD Name']].copy(), 
                               df_census_2023[["KEY",'Collection Year', 'MSA/MD', 'State', 'County', 'Census Tract','MSA/MD Name']].copy(),
                               join_columns=["KEY"],
                               df1_name="2024",
                               df2_name="2023"
                               )


print(df_compare.report())

DataComPy Comparison
--------------------

DataFrame Summary
-----------------

  DataFrame  Columns   Rows
0      2024        7  87275
1      2023        7  87275

Column Summary
--------------

Number of columns in common: 7
Number of columns in 2024 but not in 2023: 0
Number of columns in 2023 but not in 2024: 0

Row Summary
-----------

Matched on: key
Any duplicates on match values: No
Absolute Tolerance: 0
Relative Tolerance: 0
Number of rows in common: 87,275
Number of rows in 2024 but not in 2023: 0
Number of rows in 2023 but not in 2024: 0

Number of rows with some compared columns unequal: 87,275
Number of rows with all compared columns equal: 0

Column Comparison
-----------------

Number of columns compared with some values unequal: 4
Number of columns compared with all values equal: 3
Total number of values which compare unequal: 88,262

Columns with Unequal Values or Types
------------------------------------

            Column 2024 dtype 2023 dtype  # Unequal  Max Diff 

In [39]:
df_all_mismatch = df_compare.all_mismatch()


In [40]:
df_census_2024_final.isna().sum()


KEY                                   0
Collection Year                       0
MSA/MD                                0
State                                 0
County                                0
Census Tract                          0
FFIEC Median Family Income            0
Population                            2
Minority Population %               133
Number of Owner Occupied Units        2
Number of 1 to 4 Family Units       133
Tract MFI                            23
Tract to MSA Income %                 0
Median Age                          133
Small County                          0
MSA/MD Name                       15351
dtype: int64

In [41]:
df_census_2024_final.columns


Index(['KEY', 'Collection Year', 'MSA/MD', 'State', 'County', 'Census Tract',
       'FFIEC Median Family Income', 'Population', 'Minority Population %',
       'Number of Owner Occupied Units', 'Number of 1 to 4 Family Units',
       'Tract MFI', 'Tract to MSA Income %', 'Median Age', 'Small County',
       'MSA/MD Name'],
      dtype='object')

In [42]:
df_census_2024_final = df_census_2024_final[[ 'Collection Year', 'MSA/MD', 'State', 'County', 'Census Tract',
       'FFIEC Median Family Income', 'Population', 'Minority Population %',
       'Number of Owner Occupied Units', 'Number of 1 to 4 Family Units',
       'Tract MFI', 'Tract to MSA Income %', 'Median Age', 'Small County',
       'MSA/MD Name']]
df_census_2024_final.head()

,Collection Year,MSA/MD,State,County,Census Tract,FFIEC Median Family Income,Population,Minority Population %,Number of Owner Occupied Units,Number of 1 to 4 Family Units,Tract MFI,Tract to MSA Income %,Median Age,Small County,MSA/MD Name
0,2024,33860,01,001,020100,74400,1775,22.48,507,710,70699,103.79,40,T,"Montgomery, AL"
1,2024,33860,01,001,020200,74400,2055,59.42,392,717,50133,73.6,48,T,"Montgomery, AL"
2,2024,33860,01,001,020300,74400,3216,30.97,967,1401,70111,102.93,44,T,"Montgomery, AL"
3,2024,33860,01,001,020400,74400,4246,17.05,1290,1598,75580,110.95,48,T,"Montgomery, AL"
4,2024,33860,01,001,020501,74400,4322,25.71,1024,1659,90879,133.41,29,T,"Montgomery, AL"


In [43]:
df_census_2024_final.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 87275 entries, 0 to 87274
Data columns (total 15 columns):
 #   Column                          Non-Null Count  Dtype 
---  ------                          --------------  ----- 
 0   Collection Year                 87275 non-null  object
 1   MSA/MD                          87275 non-null  object
 2   State                           87275 non-null  object
 3   County                          87275 non-null  object
 4   Census Tract                    87275 non-null  object
 5   FFIEC Median Family Income      87275 non-null  object
 6   Population                      87273 non-null  object
 7   Minority Population %           87142 non-null  object
 8   Number of Owner Occupied Units  87273 non-null  object
 9   Number of 1 to 4 Family Units   87142 non-null  object
 10  Tract MFI                       87252 non-null  object
 11  Tract to MSA Income %           87275 non-null  object
 12  Median Age                      87142 non-null

# Write Output

In [86]:
df_census_2024_final.to_csv("../output_FHFA/" + "ffiec_census_{year}_with_updated_schema_cols.{end}".format(year="2024", end="txt"), 
								   index=False, 
								   sep="|")

